# Solution C1 · Collinearity: unstable coefficients, stable predictions

**Type**: Break the model

## Objective

Build two nearly identical features and show with the bootstrap that their OLS coefficients are unstable
(and negatively correlated), while their sum and the predictions stay stable. Then stabilize everything with Ridge.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LinearRegression, Ridge


def generate(n, seed):
    rng = np.random.default_rng(seed)
    x1 = rng.normal(size=n)
    x2 = x1 + rng.normal(scale=0.05, size=n)         # almost a copy of x1
    x3 = rng.normal(size=n)                          # independent of the others
    X = np.column_stack([x1, x2, x3])
    y = 2 * x1 + 2 * x2 + 1 * x3 + rng.normal(size=n)
    return X, y


X, y = generate(80, seed=0)
X_new, y_new = generate(2000, seed=1)
print("correlation x1-x2:", round(np.corrcoef(X[:, 0], X[:, 1])[0, 1], 4))

## Commented solution

In [ ]:
def vif(X):
    values = []
    for j in range(X.shape[1]):
        others = np.delete(X, j, axis=1)
        r2 = LinearRegression().fit(others, X[:, j]).score(others, X[:, j])
        values.append(1 / (1 - r2))
    return np.array(values)

vif_values = vif(X)
print("VIF:", vif_values.round(1))

In [ ]:
rng = np.random.default_rng(0)
B = 300
coef_ols = np.empty((B, 3))
coef_ridge = np.empty((B, 3))
for b in range(B):
    idx = rng.integers(0, len(y), len(y))
    coef_ols[b] = LinearRegression().fit(X[idx], y[idx]).coef_
    coef_ridge[b] = Ridge(alpha=5).fit(X[idx], y[idx]).coef_

In [ ]:
std_ols = coef_ols.std(axis=0, ddof=1)
std_ridge = coef_ridge.std(axis=0, ddof=1)
std_sum_ols = (coef_ols[:, 0] + coef_ols[:, 1]).std(ddof=1)
corr_b1_b2 = np.corrcoef(coef_ols[:, 0], coef_ols[:, 1])[0, 1]

print("std OLS  :", std_ols.round(3))
print("std Ridge:", std_ridge.round(3))
print("std of b1 + b2 (OLS):", round(std_sum_ols, 3))
print("correlation between b1 and b2 over the 300 fits:", round(corr_b1_b2, 4))

In [ ]:
rmse_ols_new = np.sqrt(np.mean((y_new - LinearRegression().fit(X, y).predict(X_new)) ** 2))
rmse_ridge_new = np.sqrt(np.mean((y_new - Ridge(alpha=5).fit(X, y).predict(X_new)) ** 2))
print(f"RMSE on new data: OLS {rmse_ols_new:.3f} | Ridge {rmse_ridge_new:.3f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8), sharex=True, sharey=True)
for a, coef, name in [(ax[0], coef_ols, "OLS"), (ax[1], coef_ridge, "Ridge (alpha=5)")]:
    a.scatter(coef[:, 0], coef[:, 1], s=10, alpha=0.5)
    a.set(title=name, xlabel="coefficient of x1", ylabel="coefficient of x2")
    a.axline((0, 4), slope=-1, color="k", lw=0.8, ls="--")    # b1 + b2 = 4, the true sum
fig.tight_layout()

## Check

In [ ]:
with verify():
    check(vif_values[0] > 50 and vif_values[1] > 50, "x1 and x2 have a huge VIF",
          "the VIF of x1 and x2 should be in the hundreds")
    check(vif_values[2] < 2, "x3 has a VIF close to 1 (it is independent)", "the VIF of x3 should be close to 1")
    check(coef_ols.shape == (300, 3) and coef_ridge.shape == (300, 3), "300 fits per model",
          "coef_ols and coef_ridge must be 300 x 3 matrices")
    check(std_ols[0] > 5 * std_ridge[0] and std_ols[1] > 5 * std_ridge[1],
          "Ridge makes the coefficients of x1 and x2 much more stable",
          "the standard deviation of x1 and x2 with OLS should be at least 5 times the one with Ridge")
    check(std_sum_ols < std_ols[0] / 5,
          "the SUM of the two coefficients is stable even with OLS",
          "the standard deviation of b1 + b2 should be much smaller than that of b1")
    check(corr_b1_b2 < -0.95, "the two coefficients are almost perfectly anti-correlated",
          "over the 300 fits b1 and b2 should have a correlation close to -1")
    check(abs(rmse_ols_new - rmse_ridge_new) / rmse_ols_new < 0.05,
          "the predictions on new data are equivalent", "OLS and Ridge should predict almost the same way")
done()

## What you learned

- With `x2 ≈ x1` the model can split the total effect (4) between the two features in infinitely many ways: (2, 2), (5, -1),
  (-3, 7)... Every resampling picks a different one, so each coefficient swings a lot (standard
  deviation above 2 for a true value of 2), but their **sum** stays stable. In the plot the points all lie
  on a line b1 + b2 = constant.
- The predictions depend on the sum, not on the individual coefficients: that is why the error on new data is identical.
  Multicollinearity is not a problem for predicting; it is for **explaining**.
- Ridge picks among the infinite solutions the one with small coefficients, splitting the effect almost
  equally between the two features: the standard deviation of the two coefficients drops from about 2.3 to about 0.07 and the interpretation becomes readable again.

## Common pitfall

Looking at the p-value or the sign of a single coefficient in the presence of high VIFs. A negative coefficient for a
variable that should have a positive effect, together with a very positive coefficient for its "twin", is the
typical symptom: it is not information about the world, it is an artifact of collinearity.

## Interview question

<details><summary>The model has excellent metrics, but the coefficient of a variable has the opposite sign from the expected one. What do you check?</summary>

First the VIFs and the correlation matrix: if the variable has a strongly correlated "twin", the sign is an
artifact of the compensation between the two coefficients, not a real effect. I would repeat the estimation on bootstrap samples
to see whether the sign is unstable, and then I would merge the two variables, drop one, or use Ridge if I only need to
predict. I would not trust the interpretation of the individual coefficients.

</details>